In [1]:
# ============================================================
# CARGAR RESULTADOS DE TODOS LOS MODELOS Y APLICAR FRIEDMAN
# 10 semillas × promedio de 5 folds por semilla
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. RUTA RAÍZ
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/alejandragomezr/models-cte-net"
)

if not ROOT.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta raíz:\n{ROOT}"
    )


# ============================================================
# 2. NOMBRES DE LOS MODELOS
# Edita únicamente los nombres de la derecha si deseas
# presentarlos de otra manera en las tablas.
# ============================================================

MODEL_PATTERNS = {
    "resultados_eegnet": "EEGNet",
    "resultados_hybridtransformer_tekte": "CTE-Net",
    "resultados_imcbgt": "IMC-BGT",
    "resultados_multistream": "MultiStream",
    "resultados_shallowconvnet": "ShallowConvNet",
    "resultados_tgarnet": "T-GARNet",
}


def identify_model(path: Path) -> str:
    """
    Identifica el modelo a partir de la ruta completa del CSV.
    """
    path_lower = str(path).lower()

    for pattern, model_name in MODEL_PATTERNS.items():
        if pattern.lower() in path_lower:
            return model_name

    return path.parent.name


# ============================================================
# 3. BUSCAR AUTOMÁTICAMENTE LOS CSV
# ============================================================

csv_files = sorted(ROOT.rglob("repeated_test_results.csv"))

print(f"CSV encontrados: {len(csv_files)}\n")

for csv_path in csv_files:
    print(f"- {identify_model(csv_path):20s} -> {csv_path}")


if len(csv_files) == 0:
    raise FileNotFoundError(
        "No se encontró ningún archivo llamado "
        "'repeated_test_results.csv'."
    )


# ============================================================
# 4. CARGAR Y NORMALIZAR LOS CSV
# ============================================================

all_results = []

for csv_path in csv_files:

    model_name = identify_model(csv_path)

    df = pd.read_csv(csv_path)

    # Normalización de nombres de columnas
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("%", "percent", regex=False)
    )

    # Posibles alias
    aliases = {
        "test_accuracy": "accuracy",
        "acc": "accuracy",
        "test_recall": "recall",
        "sensitivity": "recall",
        "test_precision": "precision",
        "test_kappa": "kappa",
        "cohen_kappa": "kappa",
        "test_auc": "auc",
        "roc_auc": "auc",
        "seed_id": "seed",
        "fold_id": "fold",
    }

    df = df.rename(
        columns={
            old: new
            for old, new in aliases.items()
            if old in df.columns and new not in df.columns
        }
    )

    required = ["seed", "fold", "accuracy"]

    missing = [
        column for column in required
        if column not in df.columns
    ]

    if missing:
        print(f"\nColumnas encontradas en {model_name}:")
        print(df.columns.tolist())

        raise ValueError(
            f"En {model_name} faltan las columnas: {missing}"
        )

    df["model"] = model_name
    df["source_file"] = str(csv_path)

    all_results.append(df)


results = pd.concat(
    all_results,
    ignore_index=True,
    sort=False
)


# ============================================================
# 5. EVITAR ARCHIVOS DUPLICADOS
# ============================================================

duplicates = results.duplicated(
    subset=["model", "seed", "fold"],
    keep=False
)

if duplicates.any():
    print("\nADVERTENCIA: se encontraron resultados duplicados:")
    print(
        results.loc[
            duplicates,
            ["model", "seed", "fold", "source_file"]
        ].sort_values(["model", "seed", "fold"])
    )

    # Se conserva la primera aparición
    results = results.drop_duplicates(
        subset=["model", "seed", "fold"],
        keep="first"
    )


# ============================================================
# 6. REVISAR CUÁNTOS RESULTADOS TIENE CADA MODELO
# ============================================================

verification = (
    results
    .groupby("model")
    .agg(
        rows=("accuracy", "size"),
        n_seeds=("seed", "nunique"),
        n_folds=("fold", "nunique")
    )
    .sort_index()
)

print("\nResumen de archivos cargados:")
display(verification)


# Verificación de folds por semilla
folds_per_seed = (
    results
    .groupby(["model", "seed"])
    .size()
    .rename("n_folds")
    .reset_index()
)

incorrect_folds = folds_per_seed[
    folds_per_seed["n_folds"] != 5
]

if not incorrect_folds.empty:
    print(
        "\nADVERTENCIA: algunas semillas no tienen exactamente 5 folds:"
    )
    display(incorrect_folds)


# ============================================================
# 7. MÉTRICAS DISPONIBLES
# ============================================================

candidate_metrics = [
    "accuracy",
    "recall",
    "precision",
    "kappa",
    "auc",
    "f1",
]

metrics = [
    metric
    for metric in candidate_metrics
    if metric in results.columns
]

print("\nMétricas disponibles:")
print(metrics)


# Convertir las métricas a valores numéricos
for metric in metrics:
    results[metric] = pd.to_numeric(
        results[metric],
        errors="coerce"
    )


# ============================================================
# 8. PROMEDIAR LOS 5 FOLDS DE CADA SEMILLA
# ============================================================

seed_results = (
    results
    .groupby(["model", "seed"], as_index=False)[metrics]
    .mean()
)

print("\nResultados promedio por semilla:")
display(seed_results.head(20))


# ============================================================
# 9. FUNCIÓN PARA FRIEDMAN Y POST HOC
# ============================================================

def friedman_and_posthoc(
    seed_dataframe: pd.DataFrame,
    metric: str,
    alpha: float = 0.05
):
    """
    Friedman:
      - bloques: semillas
      - tratamientos: modelos

    Post hoc:
      - Wilcoxon signed-rank por pares
      - corrección de Holm
    """

    pivot = seed_dataframe.pivot(
        index="seed",
        columns="model",
        values=metric
    ).sort_index()

    # Solo se conservan semillas presentes en todos los modelos
    complete = pivot.dropna(axis=0, how="any")

    if complete.shape[1] < 3:
        raise ValueError(
            f"Friedman necesita al menos tres modelos. "
            f"Para {metric} hay {complete.shape[1]}."
        )

    if complete.shape[0] < 2:
        raise ValueError(
            f"No hay suficientes semillas comunes para {metric}."
        )

    models = complete.columns.tolist()

    statistic, p_value = friedmanchisquare(
        *[complete[model].to_numpy() for model in models]
    )

    # Rango 1 = mejor resultado
    ranks = complete.rank(
        axis=1,
        method="average",
        ascending=False
    )

    mean_ranks = (
        ranks.mean(axis=0)
        .sort_values()
        .rename("mean_rank")
        .reset_index()
    )

    # Resumen descriptivo
    summary = pd.DataFrame({
        "model": models,
        "mean": [complete[m].mean() for m in models],
        "std": [complete[m].std(ddof=1) for m in models],
        "median": [complete[m].median() for m in models],
        "mean_rank": [
            ranks[m].mean()
            for m in models
        ],
        "n_seeds": complete.shape[0],
    }).sort_values("mean_rank")

    # Comparaciones por pares
    comparisons = []

    for model_a, model_b in combinations(models, 2):

        values_a = complete[model_a].to_numpy()
        values_b = complete[model_b].to_numpy()

        differences = values_a - values_b

        # Wilcoxon falla cuando todas las diferencias son cero
        if np.allclose(differences, 0):
            w_statistic = 0.0
            raw_p = 1.0
        else:
            w_statistic, raw_p = wilcoxon(
                values_a,
                values_b,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

        comparisons.append({
            "model_a": model_a,
            "model_b": model_b,
            "wilcoxon_W": w_statistic,
            "p_raw": raw_p,
            "mean_a": values_a.mean(),
            "mean_b": values_b.mean(),
            "mean_difference_a_minus_b": differences.mean(),
        })

    posthoc = pd.DataFrame(comparisons)

    reject, p_holm, _, _ = multipletests(
        posthoc["p_raw"],
        alpha=alpha,
        method="holm"
    )

    posthoc["p_holm"] = p_holm
    posthoc["significant_holm"] = reject

    posthoc = posthoc.sort_values(
        ["p_holm", "p_raw"]
    ).reset_index(drop=True)

    friedman_result = {
        "metric": metric,
        "n_seeds": complete.shape[0],
        "n_models": complete.shape[1],
        "statistic": statistic,
        "p_value": p_value,
        "significant": p_value < alpha,
    }

    return (
        friedman_result,
        summary,
        mean_ranks,
        posthoc,
        complete
    )


# ============================================================
# 10. EJECUTAR FRIEDMAN PARA TODAS LAS MÉTRICAS
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/friedman_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

global_results = []

for metric in metrics:

    print("\n" + "=" * 75)
    print(f"MÉTRICA: {metric.upper()}")
    print("=" * 75)

    (
        friedman_result,
        descriptive_summary,
        mean_ranks,
        posthoc,
        paired_data
    ) = friedman_and_posthoc(
        seed_results,
        metric=metric,
        alpha=0.05
    )

    global_results.append(friedman_result)

    print(
        f"\nFriedman χ²({friedman_result['n_models'] - 1}) = "
        f"{friedman_result['statistic']:.4f}"
    )

    print(
        f"p = {friedman_result['p_value']:.8f}"
    )

    if friedman_result["significant"]:
        print(
            "Resultado: existen diferencias significativas "
            "entre los modelos."
        )
    else:
        print(
            "Resultado: no se encontraron diferencias "
            "significativas entre los modelos."
        )

    print("\nResumen y rangos promedio:")
    display(descriptive_summary)

    print("\nPost hoc Wilcoxon con corrección de Holm:")
    display(posthoc)

    # Guardar archivos
    descriptive_summary.to_csv(
        OUTPUT_DIR / f"{metric}_summary.csv",
        index=False
    )

    mean_ranks.to_csv(
        OUTPUT_DIR / f"{metric}_mean_ranks.csv",
        index=False
    )

    posthoc.to_csv(
        OUTPUT_DIR / f"{metric}_wilcoxon_holm.csv",
        index=False
    )

    paired_data.to_csv(
        OUTPUT_DIR / f"{metric}_paired_by_seed.csv"
    )


friedman_table = pd.DataFrame(global_results)

friedman_table.to_csv(
    OUTPUT_DIR / "friedman_global_results.csv",
    index=False
)

results.to_csv(
    OUTPUT_DIR / "all_fold_results.csv",
    index=False
)

seed_results.to_csv(
    OUTPUT_DIR / "mean_results_by_seed.csv",
    index=False
)


print("\n" + "=" * 75)
print("RESUMEN GLOBAL DE FRIEDMAN")
print("=" * 75)

display(friedman_table)

print(f"\nResultados guardados en:\n{OUTPUT_DIR}")


CSV encontrados: 6

- EEGNet               -> /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_eegnet_tdah_ARTICULO-20260715T223815Z-1-001/resultados_eegnet_tdah_ARTICULO/repeated_test_results.csv
- CTE-Net              -> /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_hybridtransformer_tekte_tdah_ARTICULO-20260715T225138Z-1-001/resultados_hybridtransformer_tekte_tdah_ARTICULO/repeated_test_results.csv
- IMC-BGT              -> /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_imcbgt_tdah_ARTICULO-20260715T223804Z-1-001/resultados_imcbgt_tdah_ARTICULO/repeated_test_results.csv
- MultiStream          -> /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_multistream_tdah_ARTICULO-20260715T223803Z-1-001/resultados_multistream_tdah_ARTICULO/repeated_test_results.csv
- ShallowConvNet       -> /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_shallowconvnet_tdah_ARTICULO-20260715T223801Z-1-001/resultados_shallowcon

,rows,n_seeds,n_folds
model,,,
CTE-Net,50,10,5
EEGNet,50,10,5
IMC-BGT,50,10,5
MultiStream,50,10,5
ShallowConvNet,50,10,5
T-GARNet,50,10,5



Métricas disponibles:
['accuracy', 'recall', 'precision', 'kappa', 'auc']

Resultados promedio por semilla:


,model,seed,accuracy,recall,precision,kappa,auc
0,CTE-Net,0,0.816513,0.825447,0.844859,0.631744,0.885674
1,CTE-Net,1,0.800667,0.853895,0.804403,0.595287,0.873312
2,CTE-Net,2,0.801518,0.821654,0.827668,0.600674,0.862673
3,CTE-Net,3,0.824869,0.828606,0.856685,0.649631,0.895742
4,CTE-Net,4,0.798968,0.827720,0.819848,0.594518,0.878590
5,CTE-Net,5,0.824067,0.848848,0.844317,0.645713,0.890253
6,CTE-Net,6,0.824656,0.884593,0.812590,0.642652,0.902724
7,CTE-Net,7,0.817732,0.870989,0.818088,0.629631,0.876994
8,CTE-Net,8,0.786177,0.822793,0.807238,0.567711,0.863418
9,CTE-Net,9,0.775021,0.816694,0.787697,0.543233,0.854540



MÉTRICA: ACCURACY

Friedman χ²(5) = 46.0571
p = 0.00000001
Resultado: existen diferencias significativas entre los modelos.

Resumen y rangos promedio:


...


Post hoc Wilcoxon con corrección de Holm:


,model_a,model_b,wilcoxon_W,p_raw,mean_a,mean_b,mean_difference_a_minus_b,p_holm,significant_holm
0,CTE-Net,IMC-BGT,0.0,0.001953,0.878392,0.711645,0.166747,0.029297,True
1,CTE-Net,MultiStream,0.0,0.001953,0.878392,0.586845,0.291548,0.029297,True
2,CTE-Net,ShallowConvNet,0.0,0.001953,0.878392,0.909304,-0.030912,0.029297,True
3,CTE-Net,T-GARNet,0.0,0.001953,0.878392,0.840721,0.037671,0.029297,True
4,EEGNet,IMC-BGT,0.0,0.001953,0.888180,0.711645,0.176535,0.029297,True
5,EEGNet,MultiStream,0.0,0.001953,0.888180,0.586845,0.301335,0.029297,True
6,IMC-BGT,MultiStream,0.0,0.001953,0.711645,0.586845,0.124801,0.029297,True
7,IMC-BGT,ShallowConvNet,0.0,0.001953,0.711645,0.909304,-0.197659,0.029297,True
8,IMC-BGT,T-GARNet,0.0,0.001953,0.711645,0.840721,-0.129076,0.029297,True
9,MultiStream,ShallowConvNet,0.0,0.001953,0.586845,0.909304,-0.322459,0.029297,True



RESUMEN GLOBAL DE FRIEDMAN


,metric,n_seeds,n_models,statistic,p_value,significant
0,accuracy,10,6,46.057143,8.842042e-09,True
1,recall,10,6,33.257143,3.345857e-06,True
2,precision,10,6,47.542857,4.403035e-09,True
3,kappa,10,6,46.514286,7.135881e-09,True
4,auc,10,6,46.171429,8.380729e-09,True



Resultados guardados en:
/kaggle/working/friedman_results
